# Pre-Match strength ranking tool
Trying to determine a teams strength based on past results to predict beginning odds and favorites in a match

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import(list_available_competitions,
    load_statsbomb_matches,
    load_statsbomb_events_for_competition,
    load_understat_team_data,
    load_understat_team_data_multi,
    load_fbref_player_stats,
    _read_stat_type_with_retry,
    _flatten_and_prefix_columns,
    load_fbref_player_stats_all,
    load_football_data_odds,
    load_football_data_odds_multi)
from src.prematch_prior import (expected_score, 
    get_empirical_draw_probability, 
    get_team_elo_as_of, 
    get_prematch_prior, 
    HOME_ADVANTAGE, 
    INITIAL_ELO)
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, f1_score, roc_auc_score, auc, roc_curve, classification_report, brier_score_loss, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

In [2]:
understat_data = pd.read_parquet('../data/raw/understat_team_data.parquet')
print(understat_data.shape)
understat_data.head()

(2280, 27)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,...,away_deep_completions,home_points,home_expected_points,home_goals,home_xg,home_np_xg,home_np_xg_difference,home_ppda,home_deep_completions,league
0,1,2014,4755,2014-08-16 17:30:00,83,78,Arsenal,Crystal Palace,CRY,ARS,...,0,3,2.7946,2,1.55411,1.55411,1.395959,2.933333,9,ENG-Premier League
1,1,2014,4750,2014-08-16 15:00:00,75,72,Leicester,Everton,EVE,LEI,...,15,1,1.9461,2,1.2783,1.2783,0.665027,14.48,1,ENG-Premier League
2,1,2014,4749,2014-08-16 12:45:00,89,84,Manchester United,Swansea,SWA,MUN,...,2,0,2.2359,1,1.16635,1.16635,0.888274,10.12,8,ENG-Premier League
3,1,2014,4751,2014-08-16 15:00:00,202,91,Queens Park Rangers,Hull,HUL,QPR,...,5,0,2.0149,0,1.90067,1.12689,0.00932,12.823529,1,ENG-Premier League
4,1,2014,4752,2014-08-16 15:00:00,85,71,Stoke,Aston Villa,AVL,STO,...,4,0,0.8041,0,0.423368,0.423368,-0.486406,4.125,3,ENG-Premier League


## Data Exploration

In [3]:
print(understat_data['date'].min())
print(understat_data['date'].max())

2014-08-16 12:45:00
2016-05-17 23:00:00


In [4]:
understat_data['home_team'].unique()

<StringArray>
[             'Arsenal',            'Leicester',    'Manchester United',
  'Queens Park Rangers',                'Stoke', 'West Bromwich Albion',
             'West Ham',            'Liverpool',     'Newcastle United',
              'Burnley',          'Aston Villa',              'Chelsea',
       'Crystal Palace',              'Everton',          'Southampton',
              'Swansea',                 'Hull',           'Sunderland',
            'Tottenham',      'Manchester City',          'Bournemouth',
              'Norwich',              'Watford',              'Almeria',
              'Granada',               'Malaga',              'Sevilla',
            'Barcelona',           'Celta Vigo',                'Eibar',
              'Levante',       'Rayo Vallecano',          'Real Madrid',
               'Getafe',             'Valencia',        'Athletic Club',
      'Atletico Madrid',              'Cordoba',             'Espanyol',
  'Deportivo La Coruna',             

In [5]:
understat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2280 entries, 0 to 2279
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   league_id              2280 non-null   string        
 1   season_id              2280 non-null   Int64         
 2   game_id                2280 non-null   Int64         
 3   date                   2280 non-null   datetime64[ns]
 4   home_team_id           2280 non-null   Int64         
 5   away_team_id           2280 non-null   Int64         
 6   home_team              2280 non-null   string        
 7   away_team              2280 non-null   string        
 8   away_team_code         2280 non-null   string        
 9   home_team_code         2280 non-null   string        
 10  away_points            2280 non-null   Int64         
 11  away_expected_points   2280 non-null   Float64       
 12  away_goals             2280 non-null   Int64         
 13  awa

In [6]:
# Create a view for both home and away teams to be able to get a rolling xG/xGA and then combine the dataframes
home_view = understat_data[['game_id','date', 'home_team', 'home_xg', 'away_xg']].rename(columns={
    'home_team': 'team',
    'home_xg': 'xg_for',
    'away_xg': 'xg_against'
})
away_view = understat_data[['game_id','date', 'away_team', 'away_xg', 'home_xg']].rename(columns={
    'away_team': 'team',
    'away_xg': 'xg_for',
    'home_xg': 'xg_against'
})

In [7]:
team_match_data = pd.concat([home_view, away_view])

In [8]:
team_match_data.head()

,game_id,date,team,xg_for,xg_against
0,4755,2014-08-16 17:30:00,Arsenal,1.55411,0.158151
1,4750,2014-08-16 15:00:00,Leicester,1.2783,0.613273
2,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076
3,4751,2014-08-16 15:00:00,Queens Park Rangers,1.90067,1.11757
4,4752,2014-08-16 15:00:00,Stoke,0.423368,0.909774


In [9]:
team_match_data.groupby('team')

In [10]:
team_match_data.head()

,game_id,date,team,xg_for,xg_against
0,4755,2014-08-16 17:30:00,Arsenal,1.55411,0.158151
1,4750,2014-08-16 15:00:00,Leicester,1.2783,0.613273
2,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076
3,4751,2014-08-16 15:00:00,Queens Park Rangers,1.90067,1.11757
4,4752,2014-08-16 15:00:00,Stoke,0.423368,0.909774


### Rolling xG feature

In [11]:
# Create a rolling xG/xGA column for each team
team_match_data['rolling_xg_for'] = (
    team_match_data
    .groupby('team')['xg_for']
    .transform(lambda x: x.rolling(window=38, min_periods=1).mean())
)

In [12]:
team_match_data['rolling_xg_against'] = (
    team_match_data
    .groupby('team')['xg_against']
    .transform(lambda x: x.rolling(window=38, min_periods=1).mean())
)

In [13]:
team_match_data.head()

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against
0,4755,2014-08-16 17:30:00,Arsenal,1.55411,0.158151,1.554110,0.158151
1,4750,2014-08-16 15:00:00,Leicester,1.2783,0.613273,1.278300,0.613273
2,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076,1.166350,0.278076
3,4751,2014-08-16 15:00:00,Queens Park Rangers,1.90067,1.11757,1.900670,1.117570
4,4752,2014-08-16 15:00:00,Stoke,0.423368,0.909774,0.423368,0.909774


In [14]:
team_match_data[team_match_data['team'] == 'Manchester United']

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against
2,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076,1.166350,0.278076
38,4727,2014-09-14 16:00:00,Manchester United,2.20911,0.563524,1.687730,0.420800
55,4740,2014-09-27 15:00:00,Manchester United,0.520246,1.27452,1.298569,0.705373
67,4695,2014-10-05 12:00:00,Manchester United,1.6239,2.13558,1.379901,1.062925
87,4717,2014-10-26 16:00:00,Manchester United,1.84074,0.989916,1.472069,1.048323
...,...,...,...,...,...,...,...
667,368,2016-03-06 20:00:00,Manchester United,0.417622,0.565787,1.190536,1.113668
679,382,2016-03-20 20:00:00,Manchester United,0.900312,1.86249,1.185511,1.155775
702,402,2016-04-10 19:00:00,Manchester United,0.341238,1.79647,1.155910,1.199434
740,437,2016-05-07 15:45:00,Manchester United,1.48344,1.36825,1.163774,1.204751


Early season results will be noisier rolling estimates due to sample size; accepted as a known limitation.

Create a better way of calculating newly promoted teams by adding a no history flag

In [15]:
team_match_data = team_match_data.sort_values(['team', 'date']).reset_index(drop=True)

In [16]:
team_match_data['matches_played_before'] = team_match_data.groupby('team').cumcount()

In [17]:
# Create a threshold that willl cross over the seasons to identify newly promoted teams
ROLLING_WINDOW = 38
team_match_data['no_history_flag'] = team_match_data['matches_played_before'] < ROLLING_WINDOW

In [18]:
# Pick a team that is not newly promoted
team_match_data[team_match_data['team'] == 'Manchester United']

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against,matches_played_before,no_history_flag
2584,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076,1.166350,0.278076,0,True
2585,4767,2014-08-24 16:00:00,Manchester United,0.941557,0.666489,1.480736,0.854642,1,True
2586,4769,2014-08-30 12:45:00,Manchester United,0.467281,0.513746,1.434899,0.853332,2,True
2587,4727,2014-09-14 16:00:00,Manchester United,2.20911,0.563524,1.687730,0.420800,3,True
2588,4736,2014-09-21 13:30:00,Manchester United,1.71191,2.5422,1.466258,0.886692,4,True
...,...,...,...,...,...,...,...,...,...
2655,417,2016-04-20 23:00:00,Manchester United,1.46607,0.137416,1.488529,0.844953,71,False
2656,434,2016-05-01 17:05:00,Manchester United,1.18462,1.16622,1.480316,0.853636,72,False
2657,437,2016-05-07 15:45:00,Manchester United,1.48344,1.36825,1.163774,1.204751,73,False
2658,447,2016-05-10 23:30:00,Manchester United,0.635987,2.13311,1.135219,1.247636,74,False


In [19]:
# Pick a newly promoted team to compare
team_match_data[team_match_data['team'] == 'Leeds']

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against,matches_played_before,no_history_flag


In [20]:
# How many matches does each team have total? Anyone with very few is worth checking
team_match_data.groupby('team').size().sort_values()

team
Sporting Gijon     38
Elche              38
Real Betis         38
Cordoba            38
Hull               38
                   ..
Leicester          76
Levante            76
Liverpool          76
Manchester City    76
West Ham           76
Length: 69, dtype: int64

Made a decision to use the 2024-2025 stats as historical, can switch to a games_played_this_season computation if the training results are worse than expected.

### Home and Away split

In [21]:
# Merge in home_team from the original dataframe
team_match_data = team_match_data.merge(
    understat_data[['game_id', 'home_team']],
    on='game_id',
    how='left'
)

In [22]:
team_match_data.columns.tolist()

['game_id',
 'date',
 'team',
 'xg_for',
 'xg_against',
 'rolling_xg_for',
 'rolling_xg_against',
 'matches_played_before',
 'no_history_flag',
 'home_team']

In [23]:
# Create a is_home flag
team_match_data['is_home'] = (team_match_data['team'] == team_match_data['home_team']).astype(int)

In [24]:
team_match_data['is_home'].value_counts()

is_home
1    2280
0    2280
Name: count, dtype: int64

In [25]:
team_match_data = team_match_data.drop(columns = 'home_team')

In [26]:
# Split into two slices
home_rows = team_match_data[team_match_data['is_home'] == 1].copy()
away_rows = team_match_data[team_match_data['is_home'] == 0].copy()

In [27]:
print(len(home_rows))
print(len(away_rows))

2280
2280


In [28]:
# Compute rolling stats on Home slice
home_rows['rolling_xg_for_home'] = (
    home_rows.groupby('team')['xg_for']
    .transform(lambda x: x.rolling(window=38, min_periods=1).mean())
)

home_rows['rolling_xg_against_home'] = (
    home_rows.groupby('team')['xg_against']
    .transform(lambda x: x.rolling(window=38, min_periods=1).mean())
)

In [29]:
# Compute rolling stats on away slice
away_rows['rolling_xg_for_away'] = (
    away_rows.groupby('team')['xg_for']
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean())
)

away_rows['rolling_xg_against_away'] = (
    away_rows.groupby('team')['xg_against']
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean())
)

In [30]:
home_rows[home_rows['team'] == 'Manchester United']

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against,matches_played_before,no_history_flag,is_home,rolling_xg_for_home,rolling_xg_against_home
2584,4749,2014-08-16 12:45:00,Manchester United,1.16635,0.278076,1.166350,0.278076,0,True,1,1.166350,0.278076
2587,4727,2014-09-14 16:00:00,Manchester United,2.20911,0.563524,1.687730,0.420800,3,True,1,1.687730,0.420800
2589,4740,2014-09-27 15:00:00,Manchester United,0.520246,1.27452,1.298569,0.705373,5,True,1,1.298569,0.705373
2590,4695,2014-10-05 12:00:00,Manchester United,1.6239,2.13558,1.379901,1.062925,6,True,1,1.379901,1.062925
2592,4717,2014-10-26 16:00:00,Manchester United,1.84074,0.989916,1.472069,1.048323,8,True,1,1.472069,1.048323
2594,4661,2014-11-08 15:00:00,Manchester United,0.956776,0.400837,1.386187,0.940409,10,True,1,1.386187,0.940409
2596,4683,2014-11-29 15:00:00,Manchester United,1.76885,0.230407,1.440853,0.838980,12,True,1,1.440853,0.838980
2597,4591,2014-12-02 19:45:00,Manchester United,1.91104,1.29092,1.499626,0.895473,13,True,1,1.499626,0.895473
2599,4616,2014-12-14 13:30:00,Manchester United,1.79543,1.87114,1.532494,1.003880,15,True,1,1.532494,1.003880
2601,4636,2014-12-26 15:00:00,Manchester United,1.86147,1.03151,1.565391,1.006643,17,True,1,1.565391,1.006643


In [31]:
away_rows[away_rows['team'] == 'Manchester United']

,game_id,date,team,xg_for,xg_against,rolling_xg_for,rolling_xg_against,matches_played_before,no_history_flag,is_home,rolling_xg_for_away,rolling_xg_against_away
2585,4767,2014-08-24 16:00:00,Manchester United,0.941557,0.666489,1.480736,0.854642,1,True,0,0.941557,0.666489
2586,4769,2014-08-30 12:45:00,Manchester United,0.467281,0.513746,1.434899,0.853332,2,True,0,0.704419,0.590118
2588,4736,2014-09-21 13:30:00,Manchester United,1.71191,2.5422,1.466258,0.886692,4,True,0,1.040249,1.240812
2591,4708,2014-10-20 20:00:00,Manchester United,0.934042,0.61662,1.448104,0.846720,7,True,0,1.013698,1.084764
2593,4656,2014-11-02 13:30:00,Manchester United,0.727924,2.56723,1.418819,0.888228,9,True,0,0.956543,1.381257
2595,4675,2014-11-22 17:30:00,Manchester United,1.2791,2.00193,1.427302,0.930362,11,True,0,1.010302,1.484703
2598,4608,2014-12-08 20:00:00,Manchester United,0.799261,1.8287,1.401786,0.972422,14,True,0,0.980154,1.533845
2600,4624,2014-12-20 15:00:00,Manchester United,1.06851,0.575806,1.379614,0.953604,16,True,0,0.991198,1.414090
2602,4639,2014-12-28 12:00:00,Manchester United,2.2613,0.888,1.391874,0.927732,18,True,0,1.132321,1.355636
2603,4551,2015-01-01 12:45:00,Manchester United,0.932262,1.4082,1.367421,0.937644,19,True,0,1.112315,1.360892


In [32]:
# Join the home and away splits back to the original dataframe
home_features = home_rows[['game_id', 'rolling_xg_for_home', 'rolling_xg_against_home']]
away_features = away_rows[['game_id', 'rolling_xg_for_away', 'rolling_xg_against_away']]
model_data = understat_data.merge(home_features, on= 'game_id', how ='left')
model_data = model_data.merge(away_features, on= 'game_id', how ='left')

In [33]:
model_data.head()

,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,...,home_xg,home_np_xg,home_np_xg_difference,home_ppda,home_deep_completions,league,rolling_xg_for_home,rolling_xg_against_home,rolling_xg_for_away,rolling_xg_against_away
0,1,2014,4755,2014-08-16 17:30:00,83,78,Arsenal,Crystal Palace,CRY,ARS,...,1.55411,1.55411,1.395959,2.933333,9,ENG-Premier League,1.554110,0.158151,0.158151,1.554110
1,1,2014,4750,2014-08-16 15:00:00,75,72,Leicester,Everton,EVE,LEI,...,1.2783,1.2783,0.665027,14.48,1,ENG-Premier League,1.278300,0.613273,0.613273,1.278300
2,1,2014,4749,2014-08-16 12:45:00,89,84,Manchester United,Swansea,SWA,MUN,...,1.16635,1.16635,0.888274,10.12,8,ENG-Premier League,1.166350,0.278076,0.278076,1.166350
3,1,2014,4751,2014-08-16 15:00:00,202,91,Queens Park Rangers,Hull,HUL,QPR,...,1.90067,1.12689,0.00932,12.823529,1,ENG-Premier League,1.900670,1.117570,1.117570,1.900670
4,1,2014,4752,2014-08-16 15:00:00,85,71,Stoke,Aston Villa,AVL,STO,...,0.423368,0.423368,-0.486406,4.125,3,ENG-Premier League,0.423368,0.909774,0.909774,0.423368


In [34]:
model_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2280 entries, 0 to 2279
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   league_id                2280 non-null   string        
 1   season_id                2280 non-null   Int64         
 2   game_id                  2280 non-null   Int64         
 3   date                     2280 non-null   datetime64[ns]
 4   home_team_id             2280 non-null   Int64         
 5   away_team_id             2280 non-null   Int64         
 6   home_team                2280 non-null   string        
 7   away_team                2280 non-null   string        
 8   away_team_code           2280 non-null   string        
 9   home_team_code           2280 non-null   string        
 10  away_points              2280 non-null   Int64         
 11  away_expected_points     2280 non-null   Float64       
 12  away_goals               2280 non-

In [35]:
# Add in a target column for modeling ie result column
def get_result(row):
    if row['home_goals'] > row['away_goals']:
        return 'home_win'
    if row['home_goals'] < row['away_goals']:
        return 'away_win'
    else:
        return 'draw'

model_data['result'] = model_data.apply(get_result, axis = 1)
model_data['result'].value_counts()

result
home_win    1010
away_win     672
draw         598
Name: count, dtype: int64

## Modeling

### Logistic Regression

In [36]:
model_data = model_data.sort_values('date')

split_idx = int(len(model_data)*0.8)
train = model_data.iloc[:split_idx].copy()
test = model_data.iloc[split_idx:].copy()

In [37]:
feature_cols = ['rolling_xg_for_home', 'rolling_xg_against_home', 'rolling_xg_for_away', 'rolling_xg_against_away']

X_train = train[feature_cols]
y_train = train['result']
X_test = test[feature_cols]
y_test = test['result']


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(solver='lbfgs', multi_class='multinomial', max_iter = 100)
lr_model.fit(X_train_scaled, y_train)

accuracy = lr_model.score(X_test_scaled, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.5592


C:\Users\ausuc\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [38]:
baseline_accuracy = (test['result']=='home_win').mean()
print(f"Baseline (always predict home win): {baseline_accuracy:.4f}")

Baseline (always predict home win): 0.4693


In [39]:
print(classification_report(y_test, lr_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

    away_win       0.52      0.56      0.54       128
        draw       0.67      0.02      0.03       114
    home_win       0.57      0.85      0.68       214

    accuracy                           0.56       456
   macro avg       0.59      0.48      0.42       456
weighted avg       0.58      0.56      0.48       456



In [40]:
probs = lr_model.predict_proba(X_test_scaled)
prob_df = pd.DataFrame(probs, columns = lr_model.classes_)
print(prob_df.describe())

         away_win        draw    home_win
count  456.000000  456.000000  456.000000
mean     0.290996    0.263252    0.445752
std      0.161841    0.057560    0.187799
min      0.008530    0.033851    0.033071
25%      0.174904    0.242827    0.308710
50%      0.264490    0.276491    0.442366
75%      0.386316    0.300450    0.564073
max      0.838008    0.357851    0.957618


In [41]:
preds = lr_model.predict(X_test_scaled)
print(confusion_matrix(y_test, preds))

[[ 72   0  56]
 [ 34   2  78]
 [ 32   1 181]]


### Elo Rating System

In [42]:
# Load the needed columns into a new dataframe for an elo system
elo_matches = model_data[['game_id', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals']].copy()
elo_matches = elo_matches.sort_values('date').reset_index(drop=True)
elo_matches.head()

,game_id,date,home_team,away_team,home_goals,away_goals
0,4749,2014-08-16 12:45:00,Manchester United,Swansea,1,2
1,4750,2014-08-16 15:00:00,Leicester,Everton,2,2
2,4751,2014-08-16 15:00:00,Queens Park Rangers,Hull,0,1
3,4752,2014-08-16 15:00:00,Stoke,Aston Villa,0,1
4,4753,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,2,2


In [43]:
# Set initial elo to 1500 as the standard and load all the teams to set their starting elo at 1500
INITIAL_ELO = 1500
elo_ratings = {}

for team in pd.concat([elo_matches['home_team'], elo_matches['away_team']]).unique():
    elo_ratings[team] = INITIAL_ELO

In [44]:
print(len(elo_ratings))
print(list(elo_ratings.items())[:5])

69
[('Manchester United', 1500), ('Leicester', 1500), ('Queens Park Rangers', 1500), ('Stoke', 1500), ('West Bromwich Albion', 1500)]


Future improvements can be used to refine the elo model by building a function for each teams estimated home field advantage as opposed to using a constant of 100

In [45]:
print(expected_score(1500, 1500))  # should be exactly 0.5 -- equal ratings
print(expected_score(1600, 1500))  # should be somewhat above 0.5 -- A is favored
print(expected_score(1500, 1600))  # should be somewhat below 0.5 -- A is the underdog
print(expected_score(2000, 1000))  # should be very close to 1.0 -- A is a huge favorite

0.5
0.6400649998028851
0.35993500019711494
0.9968476908167399


In [46]:
# Establish a dynamic k factor including margin of victory 
elo_matches['margin'] = abs(elo_matches['home_goals'] - elo_matches['away_goals'])
elo_matches['k_factor'] = 30 + 30 * elo_matches['margin']
elo_matches.head(10)

,game_id,date,home_team,away_team,home_goals,away_goals,margin,k_factor
0,4749,2014-08-16 12:45:00,Manchester United,Swansea,1,2,1,60
1,4750,2014-08-16 15:00:00,Leicester,Everton,2,2,0,30
2,4751,2014-08-16 15:00:00,Queens Park Rangers,Hull,0,1,1,60
3,4752,2014-08-16 15:00:00,Stoke,Aston Villa,0,1,1,60
4,4753,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,2,2,0,30
5,4754,2014-08-16 15:00:00,West Ham,Tottenham,0,1,1,60
6,4755,2014-08-16 17:30:00,Arsenal,Crystal Palace,2,1,1,60
7,4756,2014-08-17 13:30:00,Liverpool,Southampton,2,1,1,60
8,4757,2014-08-17 16:00:00,Newcastle United,Manchester City,0,2,2,90
9,4758,2014-08-18 20:00:00,Burnley,Chelsea,1,3,2,90


In [47]:
# Sequential rating update loop
home_elo_pre = []
away_elo_pre = []
home_expected = []

for idx, row in elo_matches.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']

    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]

    # Record the ratings and expected score before the match updates
    # This is the prematch data we will use later
    home_elo_pre.append(home_rating)
    away_elo_pre.append(away_rating)

    exp_home = expected_score(home_rating, away_rating, home_advantage=HOME_ADVANTAGE)
    home_expected.append(exp_home)

    # Determine actual score: 1 = win, 0.5 = draw, 0= loss from the home teams perspective
    if row['home_goals'] > row['away_goals']:
        actual_home = 1
    elif row['home_goals']< row['away_goals']:
        actual_home = 0
    else:
        actual_home = 0.5
    actual_away = 1-actual_home

    # Update both teams ratings using the margin scaled k factor
    k= row['k_factor']
    elo_ratings[home_team] = home_rating + k * (actual_home - exp_home)
    elo_ratings[away_team] = away_rating + k * (actual_away - (1 - exp_home))

elo_matches['home_elo_pre'] = home_elo_pre
elo_matches['away_elo_pre'] = away_elo_pre
elo_matches['home_expected_score'] = home_expected

In [48]:
elo_matches[['date', 'home_team', 'away_team', 'home_elo_pre', 'away_elo_pre', 'home_expected_score']].head(10)

,date,home_team,away_team,home_elo_pre,away_elo_pre,home_expected_score
0,2014-08-16 12:45:00,Manchester United,Swansea,1500.0,1500.0,0.640065
1,2014-08-16 15:00:00,Leicester,Everton,1500.0,1500.0,0.640065
2,2014-08-16 15:00:00,Queens Park Rangers,Hull,1500.0,1500.0,0.640065
3,2014-08-16 15:00:00,Stoke,Aston Villa,1500.0,1500.0,0.640065
4,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,1500.0,1500.0,0.640065
5,2014-08-16 15:00:00,West Ham,Tottenham,1500.0,1500.0,0.640065
6,2014-08-16 17:30:00,Arsenal,Crystal Palace,1500.0,1500.0,0.640065
7,2014-08-17 13:30:00,Liverpool,Southampton,1500.0,1500.0,0.640065
8,2014-08-17 16:00:00,Newcastle United,Manchester City,1500.0,1500.0,0.640065
9,2014-08-18 20:00:00,Burnley,Chelsea,1500.0,1500.0,0.640065


In [49]:
elo_matches[['date', 'home_team', 'away_team', 'home_elo_pre', 'away_elo_pre']].tail(10)

,date,home_team,away_team,home_elo_pre,away_elo_pre
2270,2016-05-15 20:00:00,Empoli,Torino,1427.744801,1536.990019
2271,2016-05-15 20:00:00,Chievo,Bologna,1519.958424,1421.935874
2272,2016-05-15 21:00:00,Espanyol,Eibar,1381.223101,1355.554487
2273,2016-05-15 21:30:00,Sporting Gijon,Villarreal,1425.099515,1559.142208
2274,2016-05-15 21:30:00,Rayo Vallecano,Levante,1426.434992,1366.466888
2275,2016-05-15 21:30:00,Real Betis,Getafe,1451.570424,1371.905899
2276,2016-05-15 22:45:00,Udinese,Carpi,1351.313678,1448.973231
2277,2016-05-15 22:45:00,Palermo,Verona,1401.412333,1360.406727
2278,2016-05-15 22:45:00,Lazio,Fiorentina,1633.848777,1548.853656
2279,2016-05-17 23:00:00,Manchester United,Bournemouth,1616.654521,1349.188610


In [50]:
# Add a true or false column for draws
elo_matches['is_draw'] = elo_matches['home_goals'] == elo_matches['away_goals']

In [51]:
elo_matches.head()

,game_id,date,home_team,away_team,home_goals,away_goals,margin,k_factor,home_elo_pre,away_elo_pre,home_expected_score,is_draw
0,4749,2014-08-16 12:45:00,Manchester United,Swansea,1,2,1,60,1500.0,1500.0,0.640065,False
1,4750,2014-08-16 15:00:00,Leicester,Everton,2,2,0,30,1500.0,1500.0,0.640065,True
2,4751,2014-08-16 15:00:00,Queens Park Rangers,Hull,0,1,1,60,1500.0,1500.0,0.640065,False
3,4752,2014-08-16 15:00:00,Stoke,Aston Villa,0,1,1,60,1500.0,1500.0,0.640065,False
4,4753,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,2,2,0,30,1500.0,1500.0,0.640065,True


In [52]:
elo_matches['is_draw'].describe()

count      2280
unique        2
top       False
freq       1682
Name: is_draw, dtype: object

In [53]:
# Bin the home_expected_score into buckets
elo_matches['expected_score_bin'] = pd.cut(elo_matches['home_expected_score'], bins = 10)

In [54]:
elo_matches['expected_score_bin'].value_counts().sort_index()

expected_score_bin
(0.0315, 0.128]     53
(0.128, 0.224]     106
(0.224, 0.32]      133
(0.32, 0.416]      192
(0.416, 0.512]     244
(0.512, 0.608]     286
(0.608, 0.704]     374
(0.704, 0.8]       349
(0.8, 0.896]       312
(0.896, 0.992]     231
Name: count, dtype: int64

In [55]:
draw_rate_by_bin = elo_matches.groupby('expected_score_bin')['is_draw'].mean()
draw_rate_by_bin

C:\Users\ausuc\AppData\Local\Temp\ipykernel_33640\1138724089.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  draw_rate_by_bin = elo_matches.groupby('expected_score_bin')['is_draw'].mean()


expected_score_bin
(0.0315, 0.128]    0.113208
(0.128, 0.224]     0.273585
(0.224, 0.32]      0.263158
(0.32, 0.416]      0.317708
(0.416, 0.512]     0.262295
(0.512, 0.608]     0.328671
(0.608, 0.704]     0.272727
(0.704, 0.8]       0.275072
(0.8, 0.896]       0.227564
(0.896, 0.992]      0.17316
Name: is_draw, dtype: Float64

In [56]:
# Define the bin edges
elo_matches['expected_score_bin'], bin_edges = pd.cut(
    elo_matches['home_expected_score'], bins=10, retbins=True
)
print(bin_edges)

[0.03146166 0.12838464 0.22434799 0.32031134 0.41627469 0.51223804
 0.60820139 0.70416474 0.80012809 0.89609144 0.99205479]


In [57]:
# Apply the function 
elo_matches['draw_probability'] = elo_matches['home_expected_score'].apply(
    lambda x: get_empirical_draw_probability(x, bin_edges, draw_rate_by_bin)
)

In [58]:
# Split the remaining proibability into home/away win
elo_matches['prob_home_win'] = (1-elo_matches['draw_probability']) * elo_matches['home_expected_score']
elo_matches['prob_away_win'] = (1-elo_matches['draw_probability']) * (1-elo_matches['home_expected_score'])

In [59]:
elo_matches.head(10)

,game_id,date,home_team,away_team,home_goals,away_goals,margin,k_factor,home_elo_pre,away_elo_pre,home_expected_score,is_draw,expected_score_bin,draw_probability,prob_home_win,prob_away_win
0,4749,2014-08-16 12:45:00,Manchester United,Swansea,1,2,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
1,4750,2014-08-16 15:00:00,Leicester,Everton,2,2,0,30,1500.0,1500.0,0.640065,True,"(0.608, 0.704]",0.272727,0.465502,0.261771
2,4751,2014-08-16 15:00:00,Queens Park Rangers,Hull,0,1,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
3,4752,2014-08-16 15:00:00,Stoke,Aston Villa,0,1,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
4,4753,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,2,2,0,30,1500.0,1500.0,0.640065,True,"(0.608, 0.704]",0.272727,0.465502,0.261771
5,4754,2014-08-16 15:00:00,West Ham,Tottenham,0,1,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
6,4755,2014-08-16 17:30:00,Arsenal,Crystal Palace,2,1,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
7,4756,2014-08-17 13:30:00,Liverpool,Southampton,2,1,1,60,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
8,4757,2014-08-17 16:00:00,Newcastle United,Manchester City,0,2,2,90,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771
9,4758,2014-08-18 20:00:00,Burnley,Chelsea,1,3,2,90,1500.0,1500.0,0.640065,False,"(0.608, 0.704]",0.272727,0.465502,0.261771


Sanity check: is our elo system working and is the home field advantage is too little or too much

In [60]:
# Pull what we need from team match data
history_lookup = team_match_data[['game_id', 'team', 'matches_played_before']]

In [61]:
# Merge home teams history
elo_matches = elo_matches.merge(
    history_lookup,
    left_on= ['game_id','home_team'],
    right_on= ['game_id','team'],
    how='left'
)
elo_matches = elo_matches.rename(columns={'matches_played_before': 'home_matches_before'})
elo_matches = elo_matches.drop(columns = 'team')

In [62]:
# Merge away teams history
elo_matches = elo_matches.merge(
    history_lookup,
    left_on=['game_id', 'away_team'],
    right_on=['game_id', 'team'],
    how='left'
)
elo_matches = elo_matches.rename(columns={'matches_played_before': 'away_matches_before'})
elo_matches = elo_matches.drop(columns='team')

In [63]:
# Quick spot check
elo_matches[['game_id', 'home_team', 'away_team', 'home_matches_before', 'away_matches_before']].head(10)

,game_id,home_team,away_team,home_matches_before,away_matches_before
0,4749,Manchester United,Swansea,0,0
1,4750,Leicester,Everton,0,0
2,4751,Queens Park Rangers,Hull,0,0
3,4752,Stoke,Aston Villa,0,0
4,4753,West Bromwich Albion,Sunderland,0,0
5,4754,West Ham,Tottenham,0,0
6,4755,Arsenal,Crystal Palace,0,0
7,4756,Liverpool,Southampton,0,0
8,4757,Newcastle United,Manchester City,0,0
9,4758,Burnley,Chelsea,0,0


In [64]:
# Filter to only show close matches in relative strength with at least 10 games played to avoid early elo issues
experienced_matches = elo_matches[
    (elo_matches['home_matches_before'] >= 10) &
    (elo_matches['away_matches_before'] >= 10)
]

close_matches = experienced_matches[
    (experienced_matches['home_elo_pre'] - experienced_matches['away_elo_pre']).abs() <= 25
]

print(len(close_matches))

183


In [65]:
# Predicted vs actual win comparison 
close_matches = close_matches.copy()
close_matches['actual_home_win'] = close_matches['home_goals'] > close_matches['away_goals']

predicted_home_win_rate = close_matches['prob_home_win'].mean()
actual_home_win_rate = close_matches['actual_home_win'].mean()

print(f"Predicted home win rate (close matchups): {predicted_home_win_rate:.3f}")
print(f"Actual home win rate (close matchups): {actual_home_win_rate:.3f}")

Predicted home win rate (close matchups): 0.463
Actual home win rate (close matchups): 0.443


Our home field advantage seems to be fair and the 2% difference seems to be well within the normal sampling noise

####  Attempting to help solve for draw predictability issues

In [66]:
test_start_date = test['date'].min()
elo_test = elo_matches[elo_matches['date'] >= test_start_date]
elo_train = elo_matches[elo_matches['date'] < test_start_date]
print(len(elo_train), len(elo_test))

1819 461


In [67]:
# Use distance from 0.5 as a better way to show how close a match is
elo_train_distance = abs(elo_train['home_expected_score']-0.5)
iso_reg = IsotonicRegression(out_of_bounds='clip', increasing=False)
iso_reg.fit(elo_train_distance, elo_train['is_draw'])

,y_min,None
,y_max,None
,increasing,False
,out_of_bounds,'clip'


In [68]:
# Use the fitted model to generate draw probability
elo_matches['expected_score_distance'] = abs(elo_matches['home_expected_score'] - 0.5)
elo_matches['draw_probability_iso'] = iso_reg.predict(elo_matches['expected_score_distance'])

In [69]:
elo_matches[['home_expected_score', 'expected_score_distance', 'draw_probability_iso']].head(10)

,home_expected_score,expected_score_distance,draw_probability_iso
0,0.640065,0.140065,0.28877
1,0.640065,0.140065,0.28877
2,0.640065,0.140065,0.28877
3,0.640065,0.140065,0.28877
4,0.640065,0.140065,0.28877
5,0.640065,0.140065,0.28877
6,0.640065,0.140065,0.28877
7,0.640065,0.140065,0.28877
8,0.640065,0.140065,0.28877
9,0.640065,0.140065,0.28877


In [70]:
# Recompute the home/away split
elo_matches['prob_home_win_iso'] = (1 - elo_matches['draw_probability_iso']) * elo_matches['home_expected_score']
elo_matches['prob_away_win_iso'] = (1 - elo_matches['draw_probability_iso']) * (1 - elo_matches['home_expected_score'])

In [71]:
elo_matches.head(10)

,game_id,date,home_team,away_team,home_goals,away_goals,margin,k_factor,home_elo_pre,away_elo_pre,...,expected_score_bin,draw_probability,prob_home_win,prob_away_win,home_matches_before,away_matches_before,expected_score_distance,draw_probability_iso,prob_home_win_iso,prob_away_win_iso
0,4749,2014-08-16 12:45:00,Manchester United,Swansea,1,2,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
1,4750,2014-08-16 15:00:00,Leicester,Everton,2,2,0,30,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
2,4751,2014-08-16 15:00:00,Queens Park Rangers,Hull,0,1,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
3,4752,2014-08-16 15:00:00,Stoke,Aston Villa,0,1,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
4,4753,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,2,2,0,30,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
5,4754,2014-08-16 15:00:00,West Ham,Tottenham,0,1,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
6,4755,2014-08-16 17:30:00,Arsenal,Crystal Palace,2,1,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
7,4756,2014-08-17 13:30:00,Liverpool,Southampton,2,1,1,60,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
8,4757,2014-08-17 16:00:00,Newcastle United,Manchester City,0,2,2,90,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997
9,4758,2014-08-18 20:00:00,Burnley,Chelsea,1,3,2,90,1500.0,1500.0,...,"(0.608, 0.704]",0.272727,0.465502,0.261771,0,0,0.140065,0.28877,0.455233,0.255997


### Comparing the models and selecting

In [72]:
# Check to see if both test sets matchup and when the matches start
test_start_date = test['date'].min()
print(test_start_date)
print(len(test))

elo_test = elo_matches[elo_matches['date'] >= test_start_date]
print(len(elo_test))

2016-02-03 23:45:00
456
461


In [73]:
# Guarantee both models are evaluated on the exact same matches
comparison_ids = set(test['game_id']) & set(elo_matches['game_id'])
print(len(comparison_ids))

test_aligned = test[test['game_id'].isin(comparison_ids)].sort_values('game_id').reset_index(drop=True)
elo_test_aligned = elo_test[elo_test['game_id'].isin(comparison_ids)].sort_values('game_id').reset_index(drop=True)

print(len(test_aligned), len(elo_test_aligned))

456
456 456


In [74]:
# Get predicted probabilities for both models

# Logistic Regression
lr_probs = lr_model.predict_proba(scaler.transform(test_aligned[feature_cols]))
print(lr_model.classes_)

# Elo
elo_probs = elo_test_aligned[['prob_away_win','draw_probability','prob_home_win']].values

# Elo with new Isotonic Regression
elo_probs_iso = elo_test_aligned[['prob_away_win_iso', 'draw_probability_iso', 'prob_home_win_iso']].values

['away_win' 'draw' 'home_win']


In [75]:
# Brier Score
y_true_onehot = pd.get_dummies(test_aligned['result'])[['away_win', 'draw', 'home_win']]

lr_brier = ((lr_probs - y_true_onehot)**2).sum(axis=1).mean()
elo_brier_binned = ((elo_probs - y_true_onehot)**2).sum(axis=1).mean()
elo_brier_iso = ((elo_probs_iso - y_true_onehot) ** 2).sum(axis=1).mean()

print(f"Logistic Regression Brier score: {lr_brier:.4f}")
print(f"Elo (binned) Brier score: {elo_brier_binned:.4f}")
print(f"Elo (isotonic) Brier score: {elo_brier_iso:.4f}")

Logistic Regression Brier score: 0.5686
Elo (binned) Brier score: 0.5726
Elo (isotonic) Brier score: 0.5750


In [76]:
# Log Loss
lr_logloss = log_loss(test_aligned['result'], lr_probs, labels=lr_model.classes_)
elo_logloss_binned = log_loss(test_aligned['result'], elo_probs, labels=lr_model.classes_)
elo_logloss_iso = log_loss(test_aligned['result'], elo_probs_iso, labels=lr_model.classes_)

print(f"Logistic Regression log loss: {lr_logloss:.4f}")
print(f"Elo (binned) log loss: {elo_logloss_binned:.4f}")
print(f"Elo (isotonic) log loss: {elo_logloss_iso:.4f}")

Logistic Regression log loss: 0.9592
Elo (binned) log loss: 0.9681
Elo (isotonic) log loss: 0.9724


In [77]:
# Sanity check: is the elo model having the same issue predicting draws as the logistic regression model did

# Split test set by actual outcome, compare average predicted probability for the correct class
for outcome in ['home_win', 'draw', 'away_win']:
    mask = test_aligned['result'] == outcome
    class_idx = list(lr_model.classes_).index(outcome)
    print(outcome)
    print('  LR avg prob assigned to correct class:', lr_probs[mask.values, class_idx].mean())
    print('  Elo (binned) avg prob assigned to correct class:', elo_probs[mask.values, class_idx].mean())
    print('  Elo (isotonic) avg prob assigned to correct class:', elo_probs_iso[mask.values, class_idx].mean())

home_win
  LR avg prob assigned to correct class: 0.5194492592459101
  Elo (binned) avg prob assigned to correct class: 0.5385804968245324
  Elo (isotonic) avg prob assigned to correct class: 0.541788990488871
draw
  LR avg prob assigned to correct class: 0.28070522400093995
  Elo (binned) avg prob assigned to correct class: 0.2674146057928809
  Elo (isotonic) avg prob assigned to correct class: 0.2643160904425038
away_win
  LR avg prob assigned to correct class: 0.39331092907400655
  Elo (binned) avg prob assigned to correct class: 0.41793336487201005
  Elo (isotonic) avg prob assigned to correct class: 0.409568394566657


I have decided to go with the binned Elo model even though it is worse than the logistic regression at predicting a draw ultimately we want to predict wins and losses in addition to team strength which it does better than the LR model.

### get_prematch_prior function

In [78]:
# Set initial elo to 1500 as the standard and load all the teams to set their starting elo at 1500
INITIAL_ELO = 1500
elo_ratings = {}

for team in pd.concat([elo_matches['home_team'], elo_matches['away_team']]).unique():
    elo_ratings[team] = INITIAL_ELO

In [79]:

home_elo_pre = []
away_elo_pre = []
home_expected = []
home_elo_post = []
away_elo_post = []
    
for idx, row in elo_matches.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    
    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]
    
    # Record the ratings and expected score before the match updates
    # This is the prematch data we will use later
    home_elo_pre.append(home_rating)
    away_elo_pre.append(away_rating)
    
    exp_home = expected_score(home_rating, away_rating, home_advantage=HOME_ADVANTAGE)
    home_expected.append(exp_home)
    
    # Determine actual score: 1 = win, 0.5 = draw, 0= loss from the home teams perspective
    if row['home_goals'] > row['away_goals']:
        actual_home = 1
    elif row['home_goals']< row['away_goals']:
        actual_home = 0
    else:
        actual_home = 0.5
    actual_away = 1-actual_home
    
    # Update both teams ratings using the margin scaled k factor
    k= row['k_factor']
    elo_ratings[home_team] = home_rating + k * (actual_home - exp_home)
    elo_ratings[away_team] = away_rating + k * (actual_away - (1 - exp_home))

    # Record what the ratings became after this match
    home_elo_post.append(elo_ratings[home_team])
    away_elo_post.append(elo_ratings[away_team])
    
elo_matches['home_elo_pre'] = home_elo_pre
elo_matches['away_elo_pre'] = away_elo_pre
elo_matches['home_expected_score'] = home_expected
elo_matches['home_elo_post'] = home_elo_post   # NEW
elo_matches['away_elo_post'] = away_elo_post   # NEW

In [80]:
elo_matches[['date', 'home_team', 'away_team', 'home_elo_pre', 'home_elo_post']].head(5)

,date,home_team,away_team,home_elo_pre,home_elo_post
0,2014-08-16 12:45:00,Manchester United,Swansea,1500.0,1461.59610
1,2014-08-16 15:00:00,Leicester,Everton,1500.0,1495.79805
2,2014-08-16 15:00:00,Queens Park Rangers,Hull,1500.0,1461.59610
3,2014-08-16 15:00:00,Stoke,Aston Villa,1500.0,1461.59610
4,2014-08-16 15:00:00,West Bromwich Albion,Sunderland,1500.0,1495.79805


In [81]:
#Unified team rating over time
home_ratings = elo_matches[['date', 'home_team', 'home_elo_post']].rename(
    columns={'home_team': 'team', 'home_elo_post': 'elo_post'}
)
away_ratings = elo_matches[['date', 'away_team', 'away_elo_post']].rename(
    columns={'away_team': 'team', 'away_elo_post': 'elo_post'}
)

team_elo_history = pd.concat([home_ratings, away_ratings]).sort_values(['team', 'date']).reset_index(drop=True)

In [82]:
team_elo_history[team_elo_history['team'] == 'Manchester United'].head(10)

,date,team,elo_post
2584,2014-08-16 12:45:00,Manchester United,1461.596100
2585,2014-08-24 16:00:00,Manchester United,1467.428954
2586,2014-08-30 12:45:00,Manchester United,1470.006358
2587,2014-09-14 16:00:00,Manchester United,1520.037981
2588,2014-09-21 13:30:00,Manchester United,1486.483988
2589,2014-09-27 15:00:00,Manchester United,1509.697938
2590,2014-10-05 12:00:00,Manchester United,1527.526008
2591,2014-10-20 20:00:00,Manchester United,1529.614312
2592,2014-10-26 16:00:00,Manchester United,1533.934982
2593,2014-11-02 13:30:00,Manchester United,1519.405551


In [83]:
print(get_team_elo_as_of('Manchester United', pd.Timestamp('2015-01-01'), team_elo_history))
print(get_team_elo_as_of('Manchester United', pd.Timestamp('2014-08-16'), team_elo_history))  # before their first match -- should be 1500

1674.5150164384518
1500


In [84]:
get_prematch_prior('Manchester United', 'Swansea', pd.Timestamp('2015-01-01'), team_elo_history, bin_edges, draw_rate_by_bin)

{'prob_home_win': np.float64(0.653958383015857),
 'prob_draw': np.float64(0.22756410256410256),
 'prob_away_win': np.float64(0.11847751442004047)}

### Save important data for later

In [86]:
team_elo_history.to_parquet('../data/processed/team_elo_history.parquet')
np.save('../data/processed/elo_bin_edges.npy', bin_edges)
draw_rate_by_bin.to_pickle('../data/processed/draw_rate_by_bin.pkl')

In [87]:
test_load = pd.read_parquet('../data/processed/team_elo_history.parquet')
test_edges = np.load('../data/processed/elo_bin_edges.npy')
test_draw_rates = pd.read_pickle('../data/processed/draw_rate_by_bin.pkl')

print(test_load.shape, len(test_edges), test_draw_rates.shape)

(4560, 3) 11 (10,)


In [88]:
import joblib
joblib.dump(lr_model, '../data/processed/lr_model.pkl')
joblib.dump(scaler, '../data/processed/lr_scaler.pkl')

['../data/processed/lr_scaler.pkl']